# 08 _ Inference Pipeline Design

**Project:** Corporacion Favorita Store Sales Forecasting  
**Project type:** Data Science / MLOps Portfolio Project  
**Notebook:** `notebooks/08_inference_pipeline_design.ipynb`

---

## 1.Purpose

This notebook validates a reproducible batch inference pipeline for the final recommended model.

The goal is to separate model training from prediction serving and prepare the project for a future API layer.


# 2.Notebook Objective

The objective of this notebook is to design and validate a serving-ready inference contract before implementing FastAPI.

The notebook will:

- Load versioned model artifacts.
- Load model-ready inference features.
- Validate raw and model feature schemas.
- Align categorical feature handling.
- Generate sales predictions.
- Validate prediction outputs.
- Save traceable inference artifacts.
- Define API-ready input/output contracts and migration notes.


# 3.Scope.

## Scope.

This notebook is a serving-readiness layer, not another modeling notebook.

It includes:

- Batch inference for the Kaggle test horizon.
- Final model artifact loading.
- Feature contract validation.
- Prediction generation and post-processing.
- Inference report creation.
- API contract design.

It does not include:

- Model training.
- Hyperparameter tuning.
- EDA.
- Data cleaning.
- Full FastAPI implementation.
- Docker implementation.
- Model selection changes.


# 4.Imports and Path Setup

This section imports the required libraries, detects the project root, loads reusable `src` utilities when available, and prepares the inference artifact paths.


In [ ]:
# Standard library
import json
import platform
import warnings
from datetime import datetime, timezone
from pathlib import Path

# Data manipulation
import numpy as np
import pandas as pd

# Notebook display
from IPython.display import display

# Model serving
try:
    import lightgbm as lgb
except ImportError as error:
    raise ImportError(
        "LightGBM is required for inference. " "Install it with: pip install lightgbm"
    ) from error


warnings.filterwarnings("ignore")

pd.set_option("display.max_columns", 200)
pd.set_option("display.max_rows", 120)
pd.set_option("display.float_format", lambda value: f"{value:,.4f}")

SEED = 42
np.random.seed(SEED)


def find_project_root(
    start_path=None,
    required_dirs=("data", "notebooks", "reports", "models"),
):
    """Find the project root by walking upward from the current directory."""
    current_path = Path.cwd() if start_path is None else Path(start_path)
    current_path = current_path.resolve()

    candidate_paths = [current_path] + list(current_path.parents)

    for candidate_path in candidate_paths:
        if all((candidate_path / directory).exists() for directory in required_dirs):
            return candidate_path

    raise FileNotFoundError(
        "Project root could not be detected. "
        f"Expected a parent directory containing: {required_dirs}. "
        f"Current working directory: {current_path}"
    )


def file_size_mb(path):
    """Return file size in MB if the file exists."""
    path = Path(path)

    if not path.exists():
        return np.nan

    return round(path.stat().st_size / (1024**2), 4)


def dataframe_memory_mb(df):
    """Return DataFrame memory usage in MB."""
    return round(df.memory_usage(deep=True).sum() / (1024**2), 4)


def relative_to_project(path):
    """Return a project-relative path when possible."""
    path = Path(path)

    try:
        return str(path.relative_to(PROJECT_ROOT))
    except ValueError:
        return str(path)


def utc_now_iso():
    """Return the current UTC timestamp as ISO string."""
    return datetime.now(timezone.utc).isoformat(timespec="seconds")


PROJECT_ROOT = find_project_root()

DATA_DIR = PROJECT_ROOT / "data"
RAW_DIR = DATA_DIR / "raw"
INTERIM_DIR = DATA_DIR / "interim"
PROCESSED_DIR = DATA_DIR / "processed"
FEATURES_DIR = DATA_DIR / "features"
PREDICTIONS_DIR = DATA_DIR / "predictions"

MODELS_DIR = PROJECT_ROOT / "models"
BASELINES_MODELS_DIR = MODELS_DIR / "baselines"
LIGHTGBM_MODELS_DIR = MODELS_DIR / "lightgbm"

REPORTS_DIR = PROJECT_ROOT / "reports"
REPORTS_TABLES_DIR = REPORTS_DIR / "tables"
REPORTS_INFERENCE_DIR = REPORTS_DIR / "inference"

MLFLOW_TRACKING_DIR = PROJECT_ROOT / "mlruns"
MLFLOW_EXPERIMENT_NAME = "store_sales_forecasting"

PREDICTIONS_DIR.mkdir(parents=True, exist_ok=True)
REPORTS_TABLES_DIR.mkdir(parents=True, exist_ok=True)
REPORTS_INFERENCE_DIR.mkdir(parents=True, exist_ok=True)

setup_summary = pd.DataFrame(
    [
        {"setting": "project_root", "value": str(PROJECT_ROOT)},
        {"setting": "python_version", "value": platform.python_version()},
        {"setting": "pandas_version", "value": pd.__version__},
        {"setting": "numpy_version", "value": np.__version__},
        {"setting": "lightgbm_version", "value": lgb.__version__},
        {"setting": "operating_system", "value": platform.platform()},
        {
            "setting": "reports_inference_dir",
            "value": relative_to_project(REPORTS_INFERENCE_DIR),
        },
    ]
)

display(setup_summary)

print("Section conclusion: inference environment and project paths are configured.")


,setting,value
0,project_root,.
1,python_version,3.11.9
2,pandas_version,2.3.3
3,numpy_version,2.4.5
4,lightgbm_version,4.6.0
5,operating_system,Windows-10-10.0.19045-SP0
6,reports_inference_dir,reports\inference


Section conclusion: inference environment and project paths are configured.


# 5.Inference contract and artifact configuration

This section defines the batch inference contract and the artifacts required to serve the final model.

The selected operational model remains `lightgbm_v1`.

If the full model artifact exists, the notebook uses it for serving because it was trained on the full available historical dataset for submission generation.

If the full model artifact does not exist, the notebook falls back to the validation-trained `lightgbm_v1` model and records that decision in the inference metadata.

### Section conclusion

The inference pipeline uses explicit versioned artifacts and does not depend on manual notebook state.


In [ ]:
# Model versions
FINAL_RECOMMENDED_MODEL_VERSION = "lightgbm_v1"
FULL_SERVING_MODEL_VERSION = "lightgbm_v1_full"

# Forecasting contract
PREDICTION_GRAIN = ["date", "store_nbr", "family"]
TARGET_COLUMN = "sales"
PREDICTION_COLUMN = "prediction"
EXPECTED_FORECAST_HORIZON_DAYS = 16

# Batch inference input
BASELINE_TEST_FEATURES_PATH = FEATURES_DIR / "baseline_test_features.parquet"
BASELINE_TRAIN_FEATURES_PATH = FEATURES_DIR / "baseline_train_features.parquet"
BASELINE_VALID_FEATURES_PATH = FEATURES_DIR / "baseline_valid_features.parquet"
SAMPLE_SUBMISSION_PATH = RAW_DIR / "sample_submission.csv"

# Final model recommendation and MLflow evidence
FINAL_MODEL_RECOMMENDATION_PATH = REPORTS_TABLES_DIR / "final_model_recommendation.csv"
MLFLOW_FINAL_MODEL_TRACKING_SUMMARY_PATH = (
    REPORTS_TABLES_DIR / "mlflow_final_model_tracking_summary.csv"
)
MLFLOW_RUN_SUMMARY_PATH = REPORTS_TABLES_DIR / "mlflow_run_summary.csv"
MLFLOW_REGISTERED_ARTIFACTS_PATH = REPORTS_TABLES_DIR / "mlflow_registered_artifacts.csv"

# LightGBM base model artifacts
LIGHTGBM_MODEL_PATH = (
    LIGHTGBM_MODELS_DIR / f"{FINAL_RECOMMENDED_MODEL_VERSION}_model.txt"
)
LIGHTGBM_CONFIG_PATH = (
    LIGHTGBM_MODELS_DIR / f"{FINAL_RECOMMENDED_MODEL_VERSION}_config.json"
)
LIGHTGBM_FEATURE_LIST_PATH = (
    LIGHTGBM_MODELS_DIR / f"{FINAL_RECOMMENDED_MODEL_VERSION}_feature_list.json"
)
LIGHTGBM_EXPERIMENT_SUMMARY_PATH = (
    LIGHTGBM_MODELS_DIR / f"{FINAL_RECOMMENDED_MODEL_VERSION}_experiment_summary.json"
)

# LightGBM full serving artifacts
LIGHTGBM_FULL_MODEL_PATH = (
    LIGHTGBM_MODELS_DIR / f"{FULL_SERVING_MODEL_VERSION}_model.txt"
)
LIGHTGBM_FULL_CONFIG_PATH = (
    LIGHTGBM_MODELS_DIR / f"{FULL_SERVING_MODEL_VERSION}_config.json"
)

# Inference outputs
INFERENCE_OUTPUT_VERSION = "final_model_inference_v1"

INFERENCE_PREDICTIONS_PATH = (
    PREDICTIONS_DIR / f"{INFERENCE_OUTPUT_VERSION}_predictions.parquet"
)
INFERENCE_SUBMISSION_PATH = (
    PREDICTIONS_DIR / f"{INFERENCE_OUTPUT_VERSION}_submission.csv"
)
INFERENCE_REPORT_PATH = (
    REPORTS_INFERENCE_DIR / f"{INFERENCE_OUTPUT_VERSION}_report.json"
)
INFERENCE_FEATURE_VALIDATION_PATH = (
    REPORTS_INFERENCE_DIR / f"{INFERENCE_OUTPUT_VERSION}_feature_validation.csv"
)
INFERENCE_API_CONTRACT_PATH = (
    REPORTS_INFERENCE_DIR / f"{INFERENCE_OUTPUT_VERSION}_api_contract.json"
)
INFERENCE_RISK_REGISTER_PATH = (
    REPORTS_INFERENCE_DIR / f"{INFERENCE_OUTPUT_VERSION}_risk_register.csv"
)
INFERENCE_SRC_MIGRATION_PLAN_PATH = (
    REPORTS_INFERENCE_DIR / f"{INFERENCE_OUTPUT_VERSION}_src_migration_plan.csv"
)
INFERENCE_CLOSURE_CRITERIA_PATH = (
    REPORTS_INFERENCE_DIR / f"{INFERENCE_OUTPUT_VERSION}_closure_criteria.csv"
)

required_artifacts = {
    "baseline_test_features": BASELINE_TEST_FEATURES_PATH,
    "lightgbm_model": LIGHTGBM_MODEL_PATH,
    "lightgbm_config": LIGHTGBM_CONFIG_PATH,
    "lightgbm_feature_list": LIGHTGBM_FEATURE_LIST_PATH,
}

optional_artifacts = {
    "baseline_train_features": BASELINE_TRAIN_FEATURES_PATH,
    "baseline_valid_features": BASELINE_VALID_FEATURES_PATH,
    "sample_submission": SAMPLE_SUBMISSION_PATH,
    "final_model_recommendation": FINAL_MODEL_RECOMMENDATION_PATH,
    "mlflow_final_model_tracking_summary": MLFLOW_FINAL_MODEL_TRACKING_SUMMARY_PATH,
    "mlflow_run_summary": MLFLOW_RUN_SUMMARY_PATH,
    "mlflow_registered_artifacts": MLFLOW_REGISTERED_ARTIFACTS_PATH,
    "lightgbm_experiment_summary": LIGHTGBM_EXPERIMENT_SUMMARY_PATH,
    "lightgbm_full_model": LIGHTGBM_FULL_MODEL_PATH,
    "lightgbm_full_config": LIGHTGBM_FULL_CONFIG_PATH,
}

artifact_inventory = pd.DataFrame(
    [
        {
            "artifact": name,
            "required": True,
            "path": path,
            "relative_path": relative_to_project(path),
            "exists": path.exists(),
            "file_size_mb": file_size_mb(path),
        }
        for name, path in required_artifacts.items()
    ]
    + [
        {
            "artifact": name,
            "required": False,
            "path": path,
            "relative_path": relative_to_project(path),
            "exists": path.exists(),
            "file_size_mb": file_size_mb(path),
        }
        for name, path in optional_artifacts.items()
    ]
)

display(artifact_inventory)

missing_required_artifacts = artifact_inventory.query("required == True and exists == False")

if not missing_required_artifacts.empty:
    raise FileNotFoundError(
        "Missing required inference artifacts:\n"
        + "\n".join(
            [
                f"- {row.artifact}: {row.path}"
                for row in missing_required_artifacts.itertuples(index=False)
            ]
        )
    )

print("Section conclusion: required inference artifacts are available.")


# 6.Helper functions

This section defines reusable helper functions for loading artifacts, validating schemas, aligning categorical features and saving inference outputs.

These functions are intentionally written in a way that can later be moved to `src/inference/` and `src/features/`.

### Section conclusion

Reusable inference logic is isolated from notebook orchestration.


In [ ]:
def read_json(path, artifact_name):
    """Read a JSON artifact."""
    path = Path(path)

    with open(path, "r", encoding="utf-8") as file:
        data = json.load(file)

    print(f"Loaded {artifact_name}: {relative_to_project(path)}")
    return data


def save_json(data, path):
    """Save a JSON artifact with safe serialization."""
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)

    def make_serializable(value):
        if isinstance(value, Path):
            return str(value)
        if isinstance(value, (np.integer,)):
            return int(value)
        if isinstance(value, (np.floating,)):
            return float(value)
        if isinstance(value, np.ndarray):
            return value.tolist()
        if isinstance(value, (pd.Timestamp, datetime)):
            return value.isoformat()
        if isinstance(value, dict):
            return {str(key): make_serializable(item) for key, item in value.items()}
        if isinstance(value, (list, tuple)):
            return [make_serializable(item) for item in value]
        return value

    with open(path, "w", encoding="utf-8") as file:
        json.dump(make_serializable(data), file, indent=4, ensure_ascii=False)


def read_optional_csv(path, dataset_name):
    """Read an optional CSV artifact."""
    path = Path(path)

    if not path.exists():
        print(f"Optional artifact not found: {dataset_name}")
        return None

    df = pd.read_csv(path)
    print(f"Loaded {dataset_name}: {df.shape[0]:,} rows, {df.shape[1]:,} columns")
    return df


def read_parquet(path, dataset_name, columns=None):
    """Read a parquet dataset with a compact log."""
    path = Path(path)
    df = pd.read_parquet(path, columns=columns)

    print(
        f"Loaded {dataset_name}: "
        f"{df.shape[0]:,} rows, {df.shape[1]:,} columns, "
        f"{dataframe_memory_mb(df):,.2f} MB"
    )

    return df


def require_columns(df, required_columns, dataset_name):
    """Validate that a DataFrame contains required columns."""
    missing_columns = sorted(set(required_columns) - set(df.columns))

    if missing_columns:
        raise ValueError(
            f"{dataset_name} is missing required columns: {missing_columns}"
        )


def ensure_datetime(df, date_columns=("date",)):
    """Convert selected columns to datetime when present."""
    df = df.copy()

    for column in date_columns:
        if column in df.columns:
            df[column] = pd.to_datetime(df[column])

    return df


def clip_forecast_predictions(predictions):
    """Clip forecast predictions to avoid negative sales."""
    return np.clip(np.asarray(predictions, dtype=float), 0, None)


def get_first_existing_column(df, candidates):
    """Return the first available column from a candidate list."""
    for column in candidates:
        if column in df.columns:
            return column

    return None


def extract_recommended_model(recommendation_df):
    """Extract the recommended model name from the recommendation table."""
    if recommendation_df is None or recommendation_df.empty:
        return FINAL_RECOMMENDED_MODEL_VERSION

    candidate_columns = [
        "recommended_model",
        "selected_model",
        "final_model",
        "model",
        "model_name",
        "model_version",
    ]

    column = get_first_existing_column(recommendation_df, candidate_columns)

    if column is None:
        return FINAL_RECOMMENDED_MODEL_VERSION

    return str(recommendation_df[column].iloc[0])


def select_serving_model():
    """
    Select the serving model artifact.

    The full model is preferred when available because it is the batch-serving
    artifact trained on the full historical dataset. This does not change the
    selected model family; it only chooses the serving artifact.
    """
    if LIGHTGBM_FULL_MODEL_PATH.exists():
        return {
            "selected_model_version": FULL_SERVING_MODEL_VERSION,
            "base_model_version": FINAL_RECOMMENDED_MODEL_VERSION,
            "model_path": LIGHTGBM_FULL_MODEL_PATH,
            "config_path": LIGHTGBM_FULL_CONFIG_PATH
            if LIGHTGBM_FULL_CONFIG_PATH.exists()
            else LIGHTGBM_CONFIG_PATH,
            "selection_reason": "Full serving model artifact found.",
            "is_full_model": True,
        }

    return {
        "selected_model_version": FINAL_RECOMMENDED_MODEL_VERSION,
        "base_model_version": FINAL_RECOMMENDED_MODEL_VERSION,
        "model_path": LIGHTGBM_MODEL_PATH,
        "config_path": LIGHTGBM_CONFIG_PATH,
        "selection_reason": "Full serving model artifact not found; using base LightGBM artifact.",
        "is_full_model": False,
    }


def build_inference_features(feature_path):
    """
    Load the model-ready inference feature table.

    In the current notebook, feature engineering has already been executed in
    previous notebooks. In production, this function should be replaced by a
    feature builder that receives raw business inputs and creates the same
    model-ready columns.
    """
    inference_df = read_parquet(feature_path, "inference_features")
    inference_df = ensure_datetime(inference_df, date_columns=("date",))
    return inference_df


def load_categorical_reference_categories(categorical_columns):
    """
    Rebuild categorical category levels using the same feature datasets used
    during model development.

    This avoids category-code drift at prediction time.
    """
    available_reference_paths = [
        BASELINE_TRAIN_FEATURES_PATH,
        BASELINE_VALID_FEATURES_PATH,
        BASELINE_TEST_FEATURES_PATH,
    ]

    available_reference_paths = [
        path for path in available_reference_paths if path.exists()
    ]

    if not available_reference_paths:
        raise FileNotFoundError(
            "No reference feature datasets found to align categorical features."
        )

    category_map = {column: pd.Index([]) for column in categorical_columns}

    for path in available_reference_paths:
        reference_df = read_parquet(
            path,
            dataset_name=f"category_reference_{path.name}",
            columns=categorical_columns,
        )

        for column in categorical_columns:
            category_map[column] = category_map[column].union(
                pd.Index(reference_df[column].dropna().unique())
            )

    sorted_category_map = {}

    for column, values in category_map.items():
        try:
            sorted_category_map[column] = sorted(values.tolist())
        except TypeError:
            sorted_category_map[column] = sorted([str(value) for value in values.tolist()])

    return sorted_category_map


def apply_categorical_schema(df, category_map):
    """Apply categorical dtypes using a fixed category map."""
    df = df.copy()
    summary_records = []

    for column, categories in category_map.items():
        if column not in df.columns:
            raise ValueError(f"Categorical column missing from inference data: {column}")

        categorical_dtype = pd.CategoricalDtype(
            categories=categories,
            ordered=False,
        )

        unknown_values = sorted(
            set(df[column].dropna().unique()) - set(categories)
        )

        df[column] = df[column].astype(categorical_dtype)

        summary_records.append(
            {
                "feature": column,
                "n_categories": len(categories),
                "unknown_values": unknown_values,
                "unknown_value_count": len(unknown_values),
                "null_after_cast": int(df[column].isna().sum()),
            }
        )

    return df, pd.DataFrame(summary_records)


def validate_no_critical_failures(validation_df):
    """Raise an error if a validation table contains failed critical checks."""
    failed_checks = validation_df.query("critical == True and passed == False")

    if not failed_checks.empty:
        raise ValueError(
            "Critical inference validation checks failed:\n"
            + "\n".join(
                [
                    f"- {row.check}: {row.details}"
                    for row in failed_checks.itertuples(index=False)
                ]
            )
        )


print("Section conclusion: helper functions are ready.")


# 7.Load final model metadata

This section loads the final recommendation evidence, LightGBM configuration and feature contract.

The feature list artifact is the source of truth for inference feature columns and their order.

### Section conclusion

The notebook will use saved metadata instead of assumptions from previous notebook state.


In [ ]:
feature_list_artifact = read_json(
    LIGHTGBM_FEATURE_LIST_PATH,
    "lightgbm_feature_list",
)

base_model_config = read_json(
    LIGHTGBM_CONFIG_PATH,
    "lightgbm_config",
)

experiment_summary = (
    read_json(LIGHTGBM_EXPERIMENT_SUMMARY_PATH, "lightgbm_experiment_summary")
    if LIGHTGBM_EXPERIMENT_SUMMARY_PATH.exists()
    else None
)

final_model_recommendation = read_optional_csv(
    FINAL_MODEL_RECOMMENDATION_PATH,
    "final_model_recommendation",
)

mlflow_final_tracking_summary = read_optional_csv(
    MLFLOW_FINAL_MODEL_TRACKING_SUMMARY_PATH,
    "mlflow_final_model_tracking_summary",
)

recommended_model = extract_recommended_model(final_model_recommendation)

serving_model_metadata = select_serving_model()
serving_model_config = read_json(
    serving_model_metadata["config_path"],
    "serving_model_config",
)

expected_model_features = feature_list_artifact["model_features"]
categorical_features = feature_list_artifact["categorical_features"]
numeric_features = feature_list_artifact["numeric_features"]

metadata_summary = pd.DataFrame(
    [
        {"item": "recommended_model", "value": recommended_model},
        {
            "item": "selected_serving_model_version",
            "value": serving_model_metadata["selected_model_version"],
        },
        {
            "item": "base_model_version",
            "value": serving_model_metadata["base_model_version"],
        },
        {
            "item": "model_path",
            "value": relative_to_project(serving_model_metadata["model_path"]),
        },
        {
            "item": "config_path",
            "value": relative_to_project(serving_model_metadata["config_path"]),
        },
        {
            "item": "selection_reason",
            "value": serving_model_metadata["selection_reason"],
        },
        {"item": "target_column", "value": TARGET_COLUMN},
        {"item": "target_transformation", "value": "log1p"},
        {"item": "inverse_transformation", "value": "expm1"},
        {"item": "negative_prediction_policy", "value": "clip_to_zero"},
        {"item": "n_model_features", "value": len(expected_model_features)},
        {"item": "n_categorical_features", "value": len(categorical_features)},
        {"item": "n_numeric_features", "value": len(numeric_features)},
    ]
)

display(metadata_summary)

if recommended_model != FINAL_RECOMMENDED_MODEL_VERSION:
    raise ValueError(
        "The final recommendation does not match the expected operational model. "
        f"Expected {FINAL_RECOMMENDED_MODEL_VERSION}, got {recommended_model}."
    )

print("Section conclusion: final model metadata and feature contract are loaded.")


# 8.Load inference input data

This section loads the model-ready batch inference feature table.

For this project phase, the input is `baseline_test_features.parquet`, created by the feature engineering notebook.

In a future API, raw request data should be transformed into this same model-ready schema inside the application.

### Section conclusion

The batch inference input is loaded from a reproducible artifact.


In [ ]:
inference_features = build_inference_features(BASELINE_TEST_FEATURES_PATH)

input_overview = pd.DataFrame(
    [
        {
            "dataset": "inference_features",
            "rows": len(inference_features),
            "columns": inference_features.shape[1],
            "memory_mb": dataframe_memory_mb(inference_features),
            "start_date": (
                inference_features["date"].min()
                if "date" in inference_features.columns
                else pd.NaT
            ),
            "end_date": (
                inference_features["date"].max()
                if "date" in inference_features.columns
                else pd.NaT
            ),
            "stores": (
                inference_features["store_nbr"].nunique()
                if "store_nbr" in inference_features.columns
                else np.nan
            ),
            "families": (
                inference_features["family"].nunique()
                if "family" in inference_features.columns
                else np.nan
            ),
            "forecast_days": (
                inference_features["date"].nunique()
                if "date" in inference_features.columns
                else np.nan
            ),
        }
    ]
)

display(input_overview)
display(inference_features.head())

print("Section conclusion: model-ready inference features are available.")


# 9.Validate raw inference schema

This section validates the business-level inference schema before model feature validation.

The required prediction grain is:

`date + store_nbr + family`

For the Kaggle batch inference case, the expected horizon is 16 days.

### Section conclusion

The input must be structurally valid before feature-level validation.


In [ ]:
required_business_columns = [
    "id",
    "date",
    "store_nbr",
    "family",
    "onpromotion",
]

require_columns(
    inference_features,
    required_business_columns,
    "inference_features",
)

grain_duplicates = int(inference_features.duplicated(PREDICTION_GRAIN).sum())
id_duplicates = int(inference_features["id"].duplicated().sum())
forecast_horizon_days = int(inference_features["date"].nunique())
expected_rows_from_grain = (
    inference_features["date"].nunique()
    * inference_features["store_nbr"].nunique()
    * inference_features["family"].nunique()
)

raw_schema_validation = pd.DataFrame(
    [
        {
            "check": "required_business_columns_available",
            "passed": set(required_business_columns).issubset(
                inference_features.columns
            ),
            "critical": True,
            "details": f"Required columns: {required_business_columns}",
        },
        {
            "check": "prediction_grain_unique",
            "passed": grain_duplicates == 0,
            "critical": True,
            "details": f"Duplicated grain rows: {grain_duplicates:,}",
        },
        {
            "check": "id_unique",
            "passed": id_duplicates == 0,
            "critical": True,
            "details": f"Duplicated ids: {id_duplicates:,}",
        },
        {
            "check": "expected_forecast_horizon",
            "passed": forecast_horizon_days == EXPECTED_FORECAST_HORIZON_DAYS,
            "critical": True,
            "details": (
                f"Observed horizon days: {forecast_horizon_days}; "
                f"expected: {EXPECTED_FORECAST_HORIZON_DAYS}"
            ),
        },
        {
            "check": "complete_store_family_date_grid",
            "passed": len(inference_features) == expected_rows_from_grain,
            "critical": True,
            "details": (
                f"Observed rows: {len(inference_features):,}; "
                f"expected from grid: {expected_rows_from_grain:,}"
            ),
        },
        {
            "check": "no_null_prediction_grain",
            "passed": inference_features[PREDICTION_GRAIN].isna().sum().sum() == 0,
            "critical": True,
            "details": (
                "Nulls in prediction grain: "
                f"{int(inference_features[PREDICTION_GRAIN].isna().sum().sum()):,}"
            ),
        },
        {
            "check": "non_negative_onpromotion",
            "passed": (inference_features["onpromotion"] >= 0).all(),
            "critical": True,
            "details": (
                "Negative onpromotion values: "
                f"{int((inference_features['onpromotion'] < 0).sum()):,}"
            ),
        },
        {
            "check": "target_not_available_in_inference",
            "passed": TARGET_COLUMN not in inference_features.columns,
            "critical": True,
            "details": f"Target column present: {TARGET_COLUMN in inference_features.columns}",
        },
    ]
)

display(raw_schema_validation)

validate_no_critical_failures(raw_schema_validation)

print("Section conclusion: raw inference schema checks passed.")


# 10.Validate training vs inference feature consistency

This section validates that inference features match the model training contract.

The model feature list loaded from `lightgbm_v1_feature_list.json` is the source of truth.

The inference matrix must contain the same features and must use the same column order.

### Section conclusion

Feature consistency is the main protection against training-serving skew.


In [ ]:
missing_model_features = sorted(
    set(expected_model_features) - set(inference_features.columns)
)
extra_inference_columns = sorted(
    set(inference_features.columns) - set(expected_model_features)
)

if missing_model_features:
    feature_null_counts = pd.Series(dtype="int64")
    features_with_nulls = pd.Series(dtype="int64")
else:
    feature_null_counts = (
        inference_features[expected_model_features]
        .isna()
        .sum()
        .sort_values(ascending=False)
    )
    features_with_nulls = feature_null_counts[feature_null_counts > 0]

feature_validation = pd.DataFrame(
    [
        {
            "check": "all_model_features_available",
            "passed": len(missing_model_features) == 0,
            "critical": True,
            "details": missing_model_features,
        },
        {
            "check": "no_null_model_features",
            "passed": len(features_with_nulls) == 0,
            "critical": True,
            "details": features_with_nulls.to_dict(),
        },
        {
            "check": "categorical_features_available",
            "passed": set(categorical_features).issubset(inference_features.columns),
            "critical": True,
            "details": categorical_features,
        },
        {
            "check": "numeric_features_available",
            "passed": set(numeric_features).issubset(inference_features.columns),
            "critical": True,
            "details": f"{len(numeric_features)} numeric features expected",
        },
        {
            "check": "extra_columns_allowed_but_not_used",
            "passed": True,
            "critical": False,
            "details": (
                f"{len(extra_inference_columns)} extra columns available; "
                "they will not be passed to the model."
            ),
        },
    ]
)

display(feature_validation)

feature_validation.to_csv(INFERENCE_FEATURE_VALIDATION_PATH, index=False)

validate_no_critical_failures(feature_validation)

X_inference = inference_features.loc[:, expected_model_features].copy()

feature_order_matches = list(X_inference.columns) == expected_model_features

if not feature_order_matches:
    raise ValueError(
        "Inference feature order does not match the training feature order."
    )

feature_inventory = pd.DataFrame(
    {
        "feature": expected_model_features,
        "inference_dtype": [
            str(X_inference[column].dtype) for column in expected_model_features
        ],
        "missing_values": [
            int(X_inference[column].isna().sum()) for column in expected_model_features
        ],
        "missing_pct": [
            X_inference[column].isna().mean() for column in expected_model_features
        ],
        "is_categorical": [
            column in categorical_features for column in expected_model_features
        ],
    }
)

display(feature_inventory.head(30))

feature_consistency_passed = True

print(
    f"Inference matrix shape: {X_inference.shape[0]:,} rows, {X_inference.shape[1]:,} features"
)
print("Section conclusion: inference features match the training feature contract.")


# 11.Apply categorical feature schema

This section aligns categorical features before prediction.

The current training artifacts do not persist category levels explicitly, so this notebook rebuilds the category reference from the saved feature datasets.

In production, category levels should be saved as a dedicated model schema artifact.

### Section conclusion

Categorical features are aligned to reduce category-code drift during inference.


In [ ]:
category_map = load_categorical_reference_categories(categorical_features)

X_inference, categorical_alignment_summary = apply_categorical_schema(
    X_inference,
    category_map,
)

display(categorical_alignment_summary)

unknown_category_issues = categorical_alignment_summary.query("unknown_value_count > 0")

if not unknown_category_issues.empty:
    raise ValueError(
        "Unknown categorical values found in inference data:\n"
        + "\n".join(
            [
                f"- {row.feature}: {row.unknown_values}"
                for row in unknown_category_issues.itertuples(index=False)
            ]
        )
    )

print("Section conclusion: categorical inference schema is aligned.")


# 12.Load final model artifact

This section loads the selected LightGBM serving artifact from disk.

No model training is performed.

### Section conclusion

The inference pipeline loads a saved model artifact instead of relying on in-memory training state.


In [ ]:
selected_model_path = serving_model_metadata["model_path"]

if not selected_model_path.exists():
    raise FileNotFoundError(f"Selected model artifact not found: {selected_model_path}")

model = lgb.Booster(model_file=str(selected_model_path))

model_feature_names = model.feature_name()

model_load_summary = pd.DataFrame(
    [
        {
            "selected_model_version": serving_model_metadata["selected_model_version"],
            "base_model_version": serving_model_metadata["base_model_version"],
            "model_path": relative_to_project(selected_model_path),
            "model_file_size_mb": file_size_mb(selected_model_path),
            "model_num_trees": model.num_trees(),
            "model_num_features": model.num_feature(),
            "expected_num_features": len(expected_model_features),
            "model_feature_names_available": bool(model_feature_names),
        }
    ]
)

display(model_load_summary)

if model.num_feature() != len(expected_model_features):
    raise ValueError(
        "Model feature count does not match feature contract. "
        f"Model: {model.num_feature()}, expected: {len(expected_model_features)}"
    )

print("Section conclusion: final LightGBM serving artifact loaded successfully.")


# 13.Generate predictions

This section generates batch predictions for the inference feature table.

The model predicts on the log scale, so predictions are transformed back with `expm1`.

Negative predictions are clipped to zero according to the saved model policy.

### Section conclusion

Predictions are generated with the saved model and post-processed consistently.


In [ ]:
inference_started_at = utc_now_iso()

log_predictions = model.predict(X_inference)
raw_sales_predictions = np.expm1(log_predictions)

negative_predictions_before_clipping = int((raw_sales_predictions < 0).sum())
sales_predictions = clip_forecast_predictions(raw_sales_predictions)

inference_finished_at = utc_now_iso()

prediction_output_columns = [
    column
    for column in [
        "id",
        "date",
        "store_nbr",
        "family",
        "onpromotion",
    ]
    if column in inference_features.columns
]

inference_predictions = inference_features[prediction_output_columns].copy()

inference_predictions[PREDICTION_COLUMN] = sales_predictions
inference_predictions["model_name"] = FINAL_RECOMMENDED_MODEL_VERSION
inference_predictions["serving_model_version"] = serving_model_metadata[
    "selected_model_version"
]
inference_predictions["inference_output_version"] = INFERENCE_OUTPUT_VERSION
inference_predictions["inference_created_at"] = inference_finished_at

prediction_summary = pd.DataFrame(
    [
        {
            "rows": len(inference_predictions),
            "prediction_min": inference_predictions[PREDICTION_COLUMN].min(),
            "prediction_mean": inference_predictions[PREDICTION_COLUMN].mean(),
            "prediction_median": inference_predictions[PREDICTION_COLUMN].median(),
            "prediction_max": inference_predictions[PREDICTION_COLUMN].max(),
            "total_predicted_sales": inference_predictions[PREDICTION_COLUMN].sum(),
            "negative_predictions_before_clipping": negative_predictions_before_clipping,
            "null_predictions": int(
                inference_predictions[PREDICTION_COLUMN].isna().sum()
            ),
            "inference_started_at": inference_started_at,
            "inference_finished_at": inference_finished_at,
        }
    ]
)

display(prediction_summary)
display(inference_predictions.head())

print("Section conclusion: predictions generated and transformed to the sales scale.")


# 14.Validate prediction output

This section validates the prediction output before saving artifacts.

The output must contain one prediction per input row, no missing predictions and no negative final predictions.

### Section conclusion

Prediction outputs must be safe before they are consumed by Kaggle, dashboards or an API.


In [ ]:
prediction_grain_duplicates = int(
    inference_predictions.duplicated(PREDICTION_GRAIN).sum()
)

output_validation = pd.DataFrame(
    [
        {
            "check": "prediction_row_count_matches_input",
            "passed": len(inference_predictions) == len(inference_features),
            "critical": True,
            "details": (
                f"Predictions: {len(inference_predictions):,}; "
                f"input rows: {len(inference_features):,}"
            ),
        },
        {
            "check": "no_null_predictions",
            "passed": inference_predictions[PREDICTION_COLUMN].isna().sum() == 0,
            "critical": True,
            "details": (
                "Null predictions: "
                f"{int(inference_predictions[PREDICTION_COLUMN].isna().sum()):,}"
            ),
        },
        {
            "check": "no_negative_final_predictions",
            "passed": (inference_predictions[PREDICTION_COLUMN] < 0).sum() == 0,
            "critical": True,
            "details": (
                "Negative final predictions: "
                f"{int((inference_predictions[PREDICTION_COLUMN] < 0).sum()):,}"
            ),
        },
        {
            "check": "all_predictions_are_finite",
            "passed": np.isfinite(inference_predictions[PREDICTION_COLUMN]).all(),
            "critical": True,
            "details": "Prediction values must be finite.",
        },
        {
            "check": "prediction_grain_unique",
            "passed": prediction_grain_duplicates == 0,
            "critical": True,
            "details": f"Duplicated prediction grain rows: {prediction_grain_duplicates:,}",
        },
        {
            "check": "negative_raw_predictions_were_clipped",
            "passed": True,
            "critical": False,
            "details": (
                f"Negative predictions before clipping: "
                f"{negative_predictions_before_clipping:,}"
            ),
        },
    ]
)

display(output_validation)

validate_no_critical_failures(output_validation)

daily_prediction_summary = (
    inference_predictions.groupby("date", as_index=False)
    .agg(
        rows=(PREDICTION_COLUMN, "size"),
        predicted_total_sales=(PREDICTION_COLUMN, "sum"),
        predicted_mean_sales=(PREDICTION_COLUMN, "mean"),
        predicted_median_sales=(PREDICTION_COLUMN, "median"),
    )
    .sort_values("date")
)

display(daily_prediction_summary)

output_validation_passed = True

print("Section conclusion: prediction output validation passed.")


# 15.Save inference artifacts

This section saves traceable inference outputs:

- model-level prediction file;
- Kaggle-compatible submission file;
- JSON inference report.

The saved report records the model version, feature contract, validation status and output paths.

### Section conclusion

Inference outputs are persisted as reusable project artifacts.


In [ ]:
inference_predictions.to_parquet(INFERENCE_PREDICTIONS_PATH, index=False)

submission_saved = False

if SAMPLE_SUBMISSION_PATH.exists() and "id" in inference_predictions.columns:
    sample_submission = pd.read_csv(SAMPLE_SUBMISSION_PATH)

    submission = sample_submission[["id"]].merge(
        inference_predictions[["id", PREDICTION_COLUMN]],
        on="id",
        how="left",
    )

    submission = submission.rename(columns={PREDICTION_COLUMN: TARGET_COLUMN})

    if submission[TARGET_COLUMN].isna().any():
        raise ValueError(
            "Submission contains missing predictions after merging with sample_submission."
        )

    if (submission[TARGET_COLUMN] < 0).any():
        raise ValueError("Submission contains negative predictions.")

    submission.to_csv(INFERENCE_SUBMISSION_PATH, index=False)
    submission_saved = True
else:
    submission = None
    print("Sample submission not available. Kaggle submission file was not created.")

inference_report = {
    "project": "store_sales_forecasting",
    "notebook": "08_inference_pipeline_design.ipynb",
    "created_at": utc_now_iso(),
    "model": {
        "recommended_model": FINAL_RECOMMENDED_MODEL_VERSION,
        "serving_model_version": serving_model_metadata["selected_model_version"],
        "base_model_version": serving_model_metadata["base_model_version"],
        "model_path": relative_to_project(serving_model_metadata["model_path"]),
        "config_path": relative_to_project(serving_model_metadata["config_path"]),
        "selection_reason": serving_model_metadata["selection_reason"],
        "model_num_trees": model.num_trees(),
        "model_num_features": model.num_feature(),
    },
    "feature_contract": {
        "feature_list_path": relative_to_project(LIGHTGBM_FEATURE_LIST_PATH),
        "n_features": len(expected_model_features),
        "categorical_features": categorical_features,
        "numeric_feature_count": len(numeric_features),
        "feature_order_validated": feature_consistency_passed,
    },
    "input": {
        "inference_feature_path": relative_to_project(BASELINE_TEST_FEATURES_PATH),
        "rows": len(inference_features),
        "columns": inference_features.shape[1],
        "start_date": str(inference_features["date"].min().date()),
        "end_date": str(inference_features["date"].max().date()),
        "forecast_horizon_days": int(inference_features["date"].nunique()),
        "stores": int(inference_features["store_nbr"].nunique()),
        "families": int(inference_features["family"].nunique()),
    },
    "output": {
        "prediction_path": relative_to_project(INFERENCE_PREDICTIONS_PATH),
        "submission_path": (
            relative_to_project(INFERENCE_SUBMISSION_PATH) if submission_saved else None
        ),
        "prediction_rows": len(inference_predictions),
        "null_predictions": int(inference_predictions[PREDICTION_COLUMN].isna().sum()),
        "negative_final_predictions": int(
            (inference_predictions[PREDICTION_COLUMN] < 0).sum()
        ),
        "negative_predictions_before_clipping": negative_predictions_before_clipping,
        "total_predicted_sales": float(inference_predictions[PREDICTION_COLUMN].sum()),
    },
    "validation": {
        "raw_schema_checks_passed": bool(raw_schema_validation["passed"].all()),
        "feature_validation_passed": bool(feature_validation["passed"].all()),
        "output_validation_passed": bool(output_validation["passed"].all()),
    },
    "mlflow": {
        "tracking_directory": relative_to_project(MLFLOW_TRACKING_DIR),
        "experiment_name": MLFLOW_EXPERIMENT_NAME,
        "new_run_logged": False,
        "reason": "This notebook validates inference and does not register a new training experiment.",
    },
}

save_json(inference_report, INFERENCE_REPORT_PATH)

saved_inference_artifacts = pd.DataFrame(
    [
        {
            "artifact": "inference_predictions",
            "path": INFERENCE_PREDICTIONS_PATH,
            "exists": INFERENCE_PREDICTIONS_PATH.exists(),
            "file_size_mb": file_size_mb(INFERENCE_PREDICTIONS_PATH),
        },
        {
            "artifact": "inference_submission",
            "path": INFERENCE_SUBMISSION_PATH,
            "exists": INFERENCE_SUBMISSION_PATH.exists(),
            "file_size_mb": file_size_mb(INFERENCE_SUBMISSION_PATH),
        },
        {
            "artifact": "inference_report",
            "path": INFERENCE_REPORT_PATH,
            "exists": INFERENCE_REPORT_PATH.exists(),
            "file_size_mb": file_size_mb(INFERENCE_REPORT_PATH),
        },
        {
            "artifact": "feature_validation",
            "path": INFERENCE_FEATURE_VALIDATION_PATH,
            "exists": INFERENCE_FEATURE_VALIDATION_PATH.exists(),
            "file_size_mb": file_size_mb(INFERENCE_FEATURE_VALIDATION_PATH),
        },
    ]
)

saved_inference_artifacts["relative_path"] = saved_inference_artifacts["path"].apply(
    relative_to_project
)

display(saved_inference_artifacts)

print("Section conclusion: inference artifacts saved successfully.")


# 16.API-ready input and output contracts

This section defines the recommended future API contract.

The API should not require users to send model features directly.

Instead, the API should receive business-level inputs and let the backend build the model-ready feature table.

### Section conclusion

The future API should expose a simple business contract while hiding model feature complexity.


In [ ]:
api_contract = {
    "endpoint": "/predict",
    "method": "POST",
    "request_contract": {
        "description": "Business-level batch prediction request.",
        "required_fields": {
            "model_version": {
                "type": "string",
                "example": FINAL_RECOMMENDED_MODEL_VERSION,
                "required": False,
            },
            "prediction_horizon": {
                "type": "integer",
                "example": EXPECTED_FORECAST_HORIZON_DAYS,
                "required": False,
            },
            "items": {
                "type": "array",
                "required": True,
                "item_schema": {
                    "date": {
                        "type": "date",
                        "example": "2017-08-16",
                        "required": True,
                    },
                    "store_nbr": {
                        "type": "integer",
                        "example": 1,
                        "required": True,
                    },
                    "family": {
                        "type": "string",
                        "example": "GROCERY I",
                        "required": True,
                    },
                    "onpromotion": {
                        "type": "number",
                        "example": 12,
                        "required": True,
                    },
                },
            },
        },
    },
    "response_contract": {
        "description": "Prediction response with model metadata.",
        "fields": {
            "model_version": "string",
            "serving_model_version": "string",
            "prediction_created_at": "datetime",
            "predictions": [
                {
                    "date": "2017-08-16",
                    "store_nbr": 1,
                    "family": "GROCERY I",
                    "predicted_sales": 123.45,
                }
            ],
            "metadata": {
                "prediction_rows": "integer",
                "prediction_horizon_days": "integer",
                "status": "success",
                "warnings": [],
            },
        },
    },
    "validation_rules": [
        "date must be parseable as a date",
        "store_nbr must be a known store",
        "family must be a known product family",
        "onpromotion must be numeric and non-negative",
        "request rows must not contain duplicated date-store-family keys",
        "batch size should be limited",
        "model_version must be supported",
        "generated model features must match the saved feature contract",
    ],
    "expected_error_mapping": {
        "missing_required_fields": 400,
        "invalid_schema": 422,
        "unknown_model_version": 404,
        "feature_generation_failure": 422,
        "model_loading_failure": 500,
        "prediction_failure": 500,
    },
    "current_notebook_implementation": {
        "input_type": "batch_model_ready_features",
        "input_artifact": relative_to_project(BASELINE_TEST_FEATURES_PATH),
        "prediction_output": relative_to_project(INFERENCE_PREDICTIONS_PATH),
        "submission_output": (
            relative_to_project(INFERENCE_SUBMISSION_PATH) if submission_saved else None
        ),
    },
}

save_json(api_contract, INFERENCE_API_CONTRACT_PATH)

display(pd.DataFrame(api_contract["validation_rules"], columns=["api_validation_rule"]))

print(f"API contract saved to: {relative_to_project(INFERENCE_API_CONTRACT_PATH)}")
print("Section conclusion: API-ready input and output contracts are documented.")


# 17.Inference risks and production considerations

Forecasting inference has specific risks that are different from training-time evaluation.

This section documents the main risks before moving the logic into `src/` and FastAPI.

### Section conclusion

The inference pipeline is usable, but production serving requires explicit controls for drift, missing future data and feature consistency.


In [ ]:
inference_risk_register = pd.DataFrame(
    [
        {
            "risk": "training_serving_skew",
            "description": "Features are calculated differently during training and inference.",
            "impact": "high",
            "mitigation": "Move feature generation to shared src/features code and validate against saved feature contract.",
        },
        {
            "risk": "categorical_code_drift",
            "description": "Categorical feature levels may be encoded differently at prediction time.",
            "impact": "high",
            "mitigation": "Persist categorical levels as a dedicated schema artifact.",
        },
        {
            "risk": "target_leakage",
            "description": "Future sales could accidentally be used to build lag or rolling features.",
            "impact": "high",
            "mitigation": "Ensure inference features use only data available before the forecast start date.",
        },
        {
            "risk": "missing_future_covariates",
            "description": "Future oil, transactions or promotion data may not be available.",
            "impact": "medium",
            "mitigation": "Define fallback imputation policies and expose warnings in the API response.",
        },
        {
            "risk": "unknown_categories",
            "description": "New stores or product families may appear in production.",
            "impact": "medium",
            "mitigation": "Reject unsupported categories or route them to a fallback model.",
        },
        {
            "risk": "prediction_drift",
            "description": "Promotion intensity, customer behavior or store patterns may change over time.",
            "impact": "medium",
            "mitigation": "Monitor feature distributions and prediction aggregates over time.",
        },
        {
            "risk": "invalid_negative_predictions",
            "description": "Raw inverse-transformed predictions may become negative.",
            "impact": "low",
            "mitigation": "Clip predictions to zero and report the number of clipped rows.",
        },
    ]
)

inference_risk_register.to_csv(INFERENCE_RISK_REGISTER_PATH, index=False)

display(inference_risk_register)

print("Section conclusion: inference risks and mitigations are documented.")


# 18.Code that should move to `src/`

This notebook validates the inference workflow.

The reusable parts should be moved to Python modules before creating the FastAPI layer.

### Section conclusion

The notebook should become an orchestration layer, while reusable inference logic should live in `src/`.


In [ ]:
src_migration_plan = pd.DataFrame(
    [
        {
            "target_module": "src/config/paths.py",
            "logic_to_move": "Project root detection and artifact path definitions.",
            "priority": "high",
        },
        {
            "target_module": "src/inference/load_model.py",
            "logic_to_move": "Model artifact selection and LightGBM Booster loading.",
            "priority": "high",
        },
        {
            "target_module": "src/inference/validate_inputs.py",
            "logic_to_move": "Business schema, prediction grain and feature contract validation.",
            "priority": "high",
        },
        {
            "target_module": "src/features/build_inference_features.py",
            "logic_to_move": "Raw business input to model-ready inference feature generation.",
            "priority": "high",
        },
        {
            "target_module": "src/inference/predict.py",
            "logic_to_move": "Prediction generation, inverse transformation and clipping.",
            "priority": "high",
        },
        {
            "target_module": "src/inference/postprocess.py",
            "logic_to_move": "Prediction output formatting and metadata attachment.",
            "priority": "medium",
        },
        {
            "target_module": "src/schemas/inference_schema.py",
            "logic_to_move": "FastAPI/Pydantic request and response schemas.",
            "priority": "medium",
        },
        {
            "target_module": "src/utils/io.py",
            "logic_to_move": "JSON, parquet and CSV loading/saving utilities.",
            "priority": "medium",
        },
    ]
)

src_migration_plan.to_csv(INFERENCE_SRC_MIGRATION_PLAN_PATH, index=False)

display(src_migration_plan)

print(
    "Section conclusion: reusable inference components are ready to be moved into src."
)


# 19.Notebook closure criteria

This section validates whether the inference notebook can be considered complete.

The notebook is complete only if:

- required artifacts exist;
- the final recommended model is LightGBM v1;
- inference input schema is valid;
- model features match the training contract;
- predictions are generated successfully;
- prediction outputs are valid;
- inference artifacts are saved;
- API contract is documented;
- migration plan to `src/` is documented.

### Section conclusion

The project is ready for the FastAPI phase only if all closure checks pass.


In [ ]:
closure_checks = pd.DataFrame(
    [
        {
            "check": "required_artifacts_available",
            "passed": missing_required_artifacts.empty,
            "details": "All required artifacts are available.",
        },
        {
            "check": "recommended_model_is_lightgbm_v1",
            "passed": recommended_model == FINAL_RECOMMENDED_MODEL_VERSION,
            "details": f"Recommended model: {recommended_model}",
        },
        {
            "check": "raw_schema_validation_passed",
            "passed": bool(raw_schema_validation["passed"].all()),
            "details": "Business-level inference schema is valid.",
        },
        {
            "check": "feature_validation_passed",
            "passed": bool(feature_validation["passed"].all()),
            "details": "Inference features match the saved model feature contract.",
        },
        {
            "check": "categorical_alignment_passed",
            "passed": unknown_category_issues.empty,
            "details": "Categorical features aligned using reference feature datasets.",
        },
        {
            "check": "model_loaded",
            "passed": model is not None,
            "details": relative_to_project(selected_model_path),
        },
        {
            "check": "prediction_output_validation_passed",
            "passed": bool(output_validation["passed"].all()),
            "details": "Prediction output is complete and valid.",
        },
        {
            "check": "inference_predictions_saved",
            "passed": INFERENCE_PREDICTIONS_PATH.exists(),
            "details": relative_to_project(INFERENCE_PREDICTIONS_PATH),
        },
        {
            "check": "inference_report_saved",
            "passed": INFERENCE_REPORT_PATH.exists(),
            "details": relative_to_project(INFERENCE_REPORT_PATH),
        },
        {
            "check": "api_contract_saved",
            "passed": INFERENCE_API_CONTRACT_PATH.exists(),
            "details": relative_to_project(INFERENCE_API_CONTRACT_PATH),
        },
        {
            "check": "risk_register_saved",
            "passed": INFERENCE_RISK_REGISTER_PATH.exists(),
            "details": relative_to_project(INFERENCE_RISK_REGISTER_PATH),
        },
        {
            "check": "src_migration_plan_saved",
            "passed": INFERENCE_SRC_MIGRATION_PLAN_PATH.exists(),
            "details": relative_to_project(INFERENCE_SRC_MIGRATION_PLAN_PATH),
        },
    ]
)

closure_checks.to_csv(INFERENCE_CLOSURE_CRITERIA_PATH, index=False)

display(closure_checks)

if not closure_checks["passed"].all():
    failed_checks = closure_checks.query("passed == False")

    raise ValueError(
        "Inference notebook closure criteria failed:\n"
        + "\n".join(
            [
                f"- {row.check}: {row.details}"
                for row in failed_checks.itertuples(index=False)
            ]
        )
    )

print("Section conclusion: all inference notebook closure criteria passed.")


# Final conclusion

This notebook validates a reproducible inference pipeline for the selected final model.

The pipeline:

- loads saved model artifacts;
- validates the batch inference input;
- validates feature consistency against the training contract;
- aligns categorical features;
- generates predictions;
- applies post-processing;
- saves traceable inference outputs;
- documents the future API contract;
- identifies reusable logic that should move to `src/`.

The project is now ready to move from notebook-based inference into a basic FastAPI serving layer.
